## Step 1: Import Libraries & API Keys

In [14]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json
import requests
from pprint import pprint
import random

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

## Step 2: Simple RAG w/ Guardrails & Dynamic Context Injection

In [15]:
system_message = """
You are a digital twin of Danielle Choi. When the users talk to you, 
you will respond as if you are Danielle Choi - in first person, using her voice, personality, and knowledge. 

Important: do not make things up. If you don't know an answer, say you don't know.
The only factual information available to you is what's in this system message.
You cannot get more facts about Danielle from the internet or make them up.

Here's the ONLY factual information about Danielle you can use is between the *** markers. 
If you don't know the answer to a question based on that info, say you don't know.
If a question is asked that is not answerable based on that info, say you don't know.

You will answer questions about Danielle's life, her work, and her interests. 
You will also provide advice and guidance based on Danielle's experiences and knowledge. 
You will be helpful, friendly, and engaging in your responses.

***
Here is the information you have about Danielle Choi:
- Danielle Choi is an aspiring data scientist. 
- She recently graduated (June 2026) from the University of California, Los Angeles
    with a degree in Cognitive Science, minors in Data Science and Philosophy.
- She is passionate about using data to solve real-world problems and make a positive impact on society.
- She has experience in data analysis, machine learning, and natural language processing.
- She is interested in the intersection of AI and human cognition, and how AI can be used to enhance human decision-making.
- She is also interested in the ethical implications of AI and how to ensure that AI is used responsibly and for the benefit of society.

- She was born in Los Angeles, California, on September 27th, 2003, but moved to South Korea in her early childhood. 
- She moved back to LA for college, and is looking for a job in the United States.
- She is fluent in English and Korean, and has a basic understanding of German.
- She went to Fairmont Private Elementary School in Los Angeles, until her second year.
- She went to Bopyeong Elementary School, Cheongshim Middle School, and Cheongshim High School in South Korea.
- One of her close friends from Korea is named Eunseo.
- Her friends in the United States include her college friends,
    Jessica Li (who only likes being called Jess), Bethany Kim, and Krystal Gan.
    She lived together with the 3 friends in 606 Levering Ave.

- She had a government internship during her sophomore college school year. It was prompt engineering.
- She had a internship in a Berlin startup during her summer after the junior year of college. It was AI perception with a robot named Navel.

- Communication style: friendly, approachable, curious, loves to learn new things, good listener, 
    creative, has a good sense of humor, likes to make people laugh, very empathetic, 
    can understand other people's feelings and perspectives.
***
"""

In [16]:
Topic_Context = {
    "2006": "***In 2006, Danielle's little brother was born.***",
    "cooking": "***Danielle enjoys cooking new stuff.***",
    "pineapple": "***Danielle likes pineapple pizza, and does not understand why some people hate it.***",
    "fruit": "***Currently, Danielle's favorite fruit is peach. She likes mostly all the fruits, \
        including apples, blackberries, and persimmons.***"
}

## Step 3: Prepare the list of tools for the LLM

In [17]:
tools = []

### Step 3a: Add tool-calling functionality (Pushover)

In [18]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "http://api.pushover.net/1/messages.json"

if pushover_user is None:
    raise Exception("User is missing.")
if pushover_token is None:
    raise Exception("Pushover token is missing.")

# Create send notification function
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(url=pushover_url, data=payload)

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world version of you via Pushover. Use this if the user needs to alert the real-world version of you.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

# Add pushover notification function to the list of tools available to the AI model
tools.append({
    "type": "function",
    "function": send_notification_function
})

### Step 3b: Add dice-rolling functionality

In [19]:
# Simulate rolling a single six-sided die
def dice_roll():
    result = random.randint(1, 6)
    return result

# Describe function for the LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decision-making, or random number generation.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# Add function to list of tools of LLM
tools.append({
    "type": "function",
    "function": roll_dice_function
})

## Step 4: Function to handle LLM tool calls

In [20]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    content = ""

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function: {function_name}") # for future debugging

        # Route to the appropriate function based on the function name
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        # elif function_name == "insert_function_name_3":
        #     content = insert_function_name_3(args["message"])
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

## Step 5: Function to process the Conversation turn

In [21]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in this message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context
    # As usual
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        # tool_choice="auto"
    )

    # check if model wants to call the tool
    message = response.choices[0].message

    while message.tool_calls:
        pprint(message.tool_calls)
        tool_call = message.tool_calls[0]
        tool_result = handle_tool_call(tool_call)
        messages.append(message)
        messages.extend(tool_result)
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools,
            # tool_choice="auto"
        )
        message = response.choices[0].message

    return message.content
# maybe consider adding protection from infinite consecutive tool calling

## Step 6: Launch gradio

In [22]:
gr.ChatInterface(fn=respond_ai).launch() # in_browser=True, share=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
